In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata

# =========================
# CONFIG
# =========================

INPUT_CSV = "PRL_ready_personas_final.csv"   # change this if your file name is different

OUTPUT_DIR = Path("outputs/family_evidence")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR.resolve())

Output folder: C:\Users\samav\OneDrive\Documents\Vamsi Documents\Masters in CS notes\GRA\PRL_Splink\outputs\family_evidence


In [4]:
def normalize_text(value):
    """
    Basic text normalization:
    - handles nulls
    - lowercases
    - removes accents
    - removes extra spaces
    """
    if pd.isna(value):
        return np.nan
    
    value = str(value).strip().lower()
    
    # remove accents
    value = unicodedata.normalize("NFD", value)
    value = "".join(ch for ch in value if unicodedata.category(ch) != "Mn")
    
    # keep letters, numbers, ñ, and spaces
    value = re.sub(r"[^a-z0-9ñ\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    
    if value == "":
        return np.nan
    
    return value


def first_token(value):
    """
    Returns first word from a name.
    Example: 'rosa guarcaya' -> 'rosa'
    """
    if pd.isna(value):
        return np.nan
    
    parts = str(value).strip().split()
    return parts[0] if len(parts) > 0 else np.nan


def last_token(value):
    """
    Returns last word from a name.
    Example: 'rosa guarcaya' -> 'guarcaya'
    """
    if pd.isna(value):
        return np.nan
    
    parts = str(value).strip().split()
    return parts[-1] if len(parts) > 1 else np.nan


def make_key(*values):
    """
    Creates a key only when all values are present.
    Example: make_key('fernando espilco', 'rosa guarcaya')
    -> 'fernando espilco | rosa guarcaya'
    """
    clean_values = []
    
    for value in values:
        if pd.isna(value) or str(value).strip() == "":
            return np.nan
        clean_values.append(str(value).strip())
    
    return " | ".join(clean_values)

In [5]:
prl = pd.read_csv(INPUT_CSV)

print("PRL shape:", prl.shape)
print("\nColumns:")
print(prl.columns.tolist())

print("\nSource table counts:")
display(prl["source_table"].value_counts(dropna=False))

print("\nPersona type counts:")
display(prl["persona_type"].value_counts(dropna=False))

PRL shape: (47072, 32)

Columns:
['persona_idno', 'event_idno', 'original_identifier', 'source_event_type', 'source_table', 'event_type', 'persona_type', 'role_group', 'name', 'lastname', 'name_clean', 'lastname_clean', 'full_name_clean', 'first_name_clean', 'first_initial', 'event_date', 'event_year', 'event_place_clean', 'birth_date', 'birth_year', 'birth_place_clean', 'death_date', 'death_year', 'death_place_clean', 'resident_in_clean', 'gender', 'block_lastname_initial', 'block_lastname_firstname', 'block_last4_firstname', 'block_lastname_birthyear', 'matching_evidence_level', 'cleaning_issue_flags']

Source table counts:


C:\Users\samav\AppData\Local\Temp\ipykernel_41580\3240606796.py:1: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  prl = pd.read_csv(INPUT_CSV)


source_table
bautismos      24986
matrimonios    16691
entierros       5395
Name: count, dtype: int64


Persona type counts:


persona_type
mother               7614
father               7369
baptized             6340
witness              4249
godparent            3260
godmother            3251
godfather            3012
deceased             2120
wife                 2060
husband              2051
mother_of_wife       1459
mother_of_husband    1439
father_of_wife       1438
father_of_husband    1410
Name: count, dtype: int64

In [6]:
required_columns = [
    "event_idno",
    "event_date",
    "event_year",
    "source_table",
    "persona_type",
    "persona_idno",
    "full_name_clean"
]

missing_columns = [col for col in required_columns if col not in prl.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# event_place_clean may not always exist
if "event_place_clean" not in prl.columns:
    prl["event_place_clean"] = np.nan

print("All required columns are present.")

All required columns are present.


In [7]:
def standardize_source_table(value):
    """
    Converts source table names into simple labels:
    baptism, marriage, burial
    """
    value = normalize_text(value)
    
    if pd.isna(value):
        return np.nan
    
    if value in ["bautismos", "baptisms", "baptism"]:
        return "baptism"
    
    if value in ["matrimonios", "marriages", "marriage"]:
        return "marriage"
    
    if value in ["entierros", "burials", "burial"]:
        return "burial"
    
    return value


prl["source_standard"] = prl["source_table"].apply(standardize_source_table)

print("Standardized source table counts:")
display(prl["source_standard"].value_counts(dropna=False))

Standardized source table counts:


source_standard
baptism     24986
marriage    16691
burial       5395
Name: count, dtype: int64

In [8]:
def standardize_persona_type(value):
    """
    Converts persona_type values into consistent role names.
    This function includes common English/Spanish variants.
    """
    value = normalize_text(value)
    
    if pd.isna(value):
        return np.nan
    
    role_map = {
        # baptism child
        "baptized": "child",
        "baptised": "child",
        "child": "child",
        "bautizado": "child",
        "bautizada": "child",
        
        # parents
        "father": "father",
        "padre": "father",
        "mother": "mother",
        "madre": "mother",
        
        # marriage spouses
        "husband": "husband",
        "groom": "husband",
        "esposo": "husband",
        "marido": "husband",
        
        "wife": "wife",
        "bride": "wife",
        "esposa": "wife",
        "mujer": "wife",
        
        # burial deceased
        "deceased": "deceased",
        "dead": "deceased",
        "difunto": "deceased",
        "difunta": "deceased",
        
        # possible marriage parent roles
        "father of husband": "husband_father",
        "husband father": "husband_father",
        "father_of_husband": "husband_father",
        "groom father": "husband_father",
        
        "mother of husband": "husband_mother",
        "husband mother": "husband_mother",
        "mother_of_husband": "husband_mother",
        "groom mother": "husband_mother",
        
        "father of wife": "wife_father",
        "wife father": "wife_father",
        "father_of_wife": "wife_father",
        "bride father": "wife_father",
        
        "mother of wife": "wife_mother",
        "wife mother": "wife_mother",
        "mother_of_wife": "wife_mother",
        "bride mother": "wife_mother",
    }
    
    return role_map.get(value, value)


prl["role_standard"] = prl["persona_type"].apply(standardize_persona_type)

print("Standardized role counts:")
display(prl["role_standard"].value_counts(dropna=False))

Standardized role counts:


role_standard
mother            7614
father            7369
child             6340
witness           4249
godparent         3260
godmother         3251
godfather         3012
deceased          2120
wife              2060
husband           2051
wife_mother       1459
husband_mother    1439
wife_father       1438
husband_father    1410
Name: count, dtype: int64

In [9]:
# Clean name column
prl["full_name_clean"] = prl["full_name_clean"].apply(normalize_text)

# Clean event ID
prl["event_idno"] = prl["event_idno"].astype(str).str.strip()

# Clean persona ID
prl["persona_idno"] = prl["persona_idno"].astype(str).str.strip()
prl.loc[prl["persona_idno"].isin(["nan", "None", ""]), "persona_idno"] = np.nan

# Convert event_year safely
prl["event_year"] = pd.to_numeric(prl["event_year"], errors="coerce")

print("Cleaned core columns.")

Cleaned core columns.


In [10]:
def first_non_null(series):
    """
    Returns the first non-null value from a pandas Series.
    """
    non_null = series.dropna()
    if len(non_null) == 0:
        return np.nan
    return non_null.iloc[0]


def build_event_role_table(
    df,
    source_name,
    allowed_roles,
    output_name
):
    """
    Builds one event-level table from persona-level data.
    
    Example:
    source_name = 'baptism'
    allowed_roles = ['child', 'father', 'mother']
    
    Output:
    one row per event_idno with child_name, father_name, mother_name, etc.
    """
    
    event_cols = [
        "event_idno",
        "event_date",
        "event_year",
        "event_place_clean"
    ]
    
    base = df[
        (df["source_standard"] == source_name) &
        (df["role_standard"].isin(allowed_roles))
    ].copy()
    
    print(f"\n{output_name} source rows:", base.shape[0])
    print(f"{output_name} unique events:", base["event_idno"].nunique())
    print(f"{output_name} role counts:")
    display(base["role_standard"].value_counts(dropna=False))
    
    # Pivot names
    names = (
        base
        .pivot_table(
            index=event_cols,
            columns="role_standard",
            values="full_name_clean",
            aggfunc=first_non_null
        )
        .reset_index()
    )
    
    # Pivot persona IDs
    ids = (
        base
        .pivot_table(
            index=["event_idno"],
            columns="role_standard",
            values="persona_idno",
            aggfunc=first_non_null
        )
        .reset_index()
    )
    
    # Flatten columns
    names.columns.name = None
    ids.columns.name = None
    
    # Rename name columns
    rename_names = {role: f"{role}_name" for role in allowed_roles if role in names.columns}
    names = names.rename(columns=rename_names)
    
    # Rename ID columns
    rename_ids = {role: f"{role}_persona_id" for role in allowed_roles if role in ids.columns}
    ids = ids.rename(columns=rename_ids)
    
    # Merge names and IDs
    result = names.merge(ids, on="event_idno", how="left")
    
    print(f"{output_name} final shape:", result.shape)
    
    return result

BUILD BAPTISM FAMILY

In [11]:
baptism_roles = ["child", "father", "mother"]

baptism_family = build_event_role_table(
    df=prl,
    source_name="baptism",
    allowed_roles=baptism_roles,
    output_name="BaptismFamily"
)

# Add first and last name parts
for role in ["child", "father", "mother"]:
    name_col = f"{role}_name"
    if name_col in baptism_family.columns:
        baptism_family[f"{role}_first_name"] = baptism_family[name_col].apply(first_token)
        baptism_family[f"{role}_last_name"] = baptism_family[name_col].apply(last_token)

# Professor-relevant keys
baptism_family["parent_pair_key"] = baptism_family.apply(
    lambda x: make_key(x.get("father_name"), x.get("mother_name")),
    axis=1
)

baptism_family["child_parent_key"] = baptism_family.apply(
    lambda x: make_key(x.get("child_name"), x.get("father_name"), x.get("mother_name")),
    axis=1
)

baptism_family["father_motherfirst_key"] = baptism_family.apply(
    lambda x: make_key(x.get("father_name"), x.get("mother_first_name")),
    axis=1
)

baptism_family["mother_fatherfirst_key"] = baptism_family.apply(
    lambda x: make_key(x.get("mother_name"), x.get("father_first_name")),
    axis=1
)

# Useful flags
baptism_family["has_child"] = baptism_family["child_name"].notna()
baptism_family["has_father"] = baptism_family["father_name"].notna()
baptism_family["has_mother"] = baptism_family["mother_name"].notna()
baptism_family["has_parent_pair"] = baptism_family["parent_pair_key"].notna()
baptism_family["has_child_parent_key"] = baptism_family["child_parent_key"].notna()

print("BaptismFamily preview:")
display(baptism_family.head())

print("BaptismFamily coverage:")
display(baptism_family[[
    "has_child",
    "has_father",
    "has_mother",
    "has_parent_pair",
    "has_child_parent_key"
]].mean().sort_values(ascending=False))


BaptismFamily source rows: 18723
BaptismFamily unique events: 6340
BaptismFamily role counts:


role_standard
child     6340
mother    6313
father    6070
Name: count, dtype: int64

BaptismFamily final shape: (6338, 10)
BaptismFamily preview:


,event_idno,event_date,event_year,event_place_clean,child_name,father_name,mother_name,child_persona_id,father_persona_id,mother_persona_id,...,mother_last_name,parent_pair_key,child_parent_key,father_motherfirst_key,mother_fatherfirst_key,has_child,has_father,has_mother,has_parent_pair,has_child_parent_key
0,bautizo-1,1790-10-04,1790.0,pampamarca,domingo ayquipa,lucas ayquipa,sevastiana quispe,persona-1,persona-2,persona-3,...,quispe,lucas ayquipa | sevastiana quispe,domingo ayquipa | lucas ayquipa | sevastiana q...,lucas ayquipa | sevastiana,sevastiana quispe | lucas,True,True,True,True,True
1,bautizo-10,1790-11-22,1790.0,chacralla,andrea flores,francisco flores,estefa espinosa,persona-34,persona-35,persona-36,...,espinosa,francisco flores | estefa espinosa,andrea flores | francisco flores | estefa espi...,francisco flores | estefa,estefa espinosa | francisco,True,True,True,True,True
2,bautizo-100,1791-11-07,1791.0,unknown,theodoro barrientos,NaN,martina barrientos,persona-376,NaN,persona-377,...,barrientos,NaN,NaN,NaN,NaN,True,False,True,False,False
3,bautizo-1000,1804-03-14,1804.0,aucara,ventura alcari,felipe alcari,ana carrera,persona-3878,persona-3879,persona-3880,...,carrera,felipe alcari | ana carrera,ventura alcari | felipe alcari | ana carrera,felipe alcari | ana,ana carrera | felipe,True,True,True,True,True
4,bautizo-1001,1804-03-20,1804.0,aucara,mariano huaman,damian huaman,eulalia vilcacuri,persona-3882,persona-3883,persona-3884,...,vilcacuri,damian huaman | eulalia vilcacuri,mariano huaman | damian huaman | eulalia vilca...,damian huaman | eulalia,eulalia vilcacuri | damian,True,True,True,True,True


BaptismFamily coverage:


has_child               1.000000
has_mother              0.995740
has_father              0.957400
has_parent_pair         0.954875
has_child_parent_key    0.954875
dtype: float64

MARRIAGE COUPLES

In [12]:
marriage_roles = [
    "husband",
    "wife",
    "husband_father",
    "husband_mother",
    "wife_father",
    "wife_mother"
]

marriage_couples = build_event_role_table(
    df=prl,
    source_name="marriage",
    allowed_roles=marriage_roles,
    output_name="MarriageCouples"
)

# Add first and last name parts
for role in marriage_roles:
    name_col = f"{role}_name"
    if name_col in marriage_couples.columns:
        marriage_couples[f"{role}_first_name"] = marriage_couples[name_col].apply(first_token)
        marriage_couples[f"{role}_last_name"] = marriage_couples[name_col].apply(last_token)

# Main spouse keys
if "husband_name" in marriage_couples.columns and "wife_name" in marriage_couples.columns:
    marriage_couples["spouse_pair_key"] = marriage_couples.apply(
        lambda x: make_key(x.get("husband_name"), x.get("wife_name")),
        axis=1
    )
    
    marriage_couples["husband_wifefirst_key"] = marriage_couples.apply(
        lambda x: make_key(x.get("husband_name"), x.get("wife_first_name")),
        axis=1
    )
    
    marriage_couples["wife_husbandfirst_key"] = marriage_couples.apply(
        lambda x: make_key(x.get("wife_name"), x.get("husband_first_name")),
        axis=1
    )
else:
    marriage_couples["spouse_pair_key"] = np.nan
    marriage_couples["husband_wifefirst_key"] = np.nan
    marriage_couples["wife_husbandfirst_key"] = np.nan

# Optional parent-pair keys if those roles exist
if "husband_father_name" in marriage_couples.columns and "husband_mother_name" in marriage_couples.columns:
    marriage_couples["husband_parent_pair_key"] = marriage_couples.apply(
        lambda x: make_key(x.get("husband_father_name"), x.get("husband_mother_name")),
        axis=1
    )
else:
    marriage_couples["husband_parent_pair_key"] = np.nan

if "wife_father_name" in marriage_couples.columns and "wife_mother_name" in marriage_couples.columns:
    marriage_couples["wife_parent_pair_key"] = marriage_couples.apply(
        lambda x: make_key(x.get("wife_father_name"), x.get("wife_mother_name")),
        axis=1
    )
else:
    marriage_couples["wife_parent_pair_key"] = np.nan

# Useful flags
marriage_couples["has_husband"] = marriage_couples.get("husband_name", pd.Series(index=marriage_couples.index)).notna()
marriage_couples["has_wife"] = marriage_couples.get("wife_name", pd.Series(index=marriage_couples.index)).notna()
marriage_couples["has_spouse_pair"] = marriage_couples["spouse_pair_key"].notna()

print("MarriageCouples preview:")
display(marriage_couples.head())

print("MarriageCouples coverage:")
display(marriage_couples[[
    "has_husband",
    "has_wife",
    "has_spouse_pair"
]].mean().sort_values(ascending=False))


MarriageCouples source rows: 9182
MarriageCouples unique events: 1719
MarriageCouples role counts:


role_standard
husband           1719
wife              1717
wife_mother       1459
husband_mother    1439
wife_father       1438
husband_father    1410
Name: count, dtype: int64

MarriageCouples final shape: (1718, 16)
MarriageCouples preview:


,event_idno,event_date,event_year,event_place_clean,husband_name,husband_father_name,husband_mother_name,wife_name,wife_father_name,wife_mother_name,...,wife_mother_first_name,wife_mother_last_name,spouse_pair_key,husband_wifefirst_key,wife_husbandfirst_key,husband_parent_pair_key,wife_parent_pair_key,has_husband,has_wife,has_spouse_pair
0,matrimonio-1,1816-12-06,1816.0,aucara,jose manl manuel de la roca,acencio roca,leonor guerrero,juana rodrigues,pedro rodrigues,magdalena sotelo,...,magdalena,sotelo,jose manl manuel de la roca | juana rodrigues,jose manl manuel de la roca | juana,juana rodrigues | jose,acencio roca | leonor guerrero,pedro rodrigues | magdalena sotelo,True,True,True
1,matrimonio-10,1817-06-20,1817.0,ishua santa iglesia,patricio asto,mateo asto,bernarda ramos,melchora sebidola,cipriano sebedola,grabiela guamani,...,grabiela,guamani,patricio asto | melchora sebidola,patricio asto | melchora,melchora sebidola | patricio,mateo asto | bernarda ramos,cipriano sebedola | grabiela guamani,True,True,True
2,matrimonio-100,1820-08-02,1820.0,santa iglesia,pedro huallpatuiru,NaN,eulalia sanches,eugenia barrientos,fernando barrientos,nolberta puma,...,nolberta,puma,pedro huallpatuiru | eugenia barrientos,pedro huallpatuiru | eugenia,eugenia barrientos | pedro,NaN,fernando barrientos | nolberta puma,True,True,True
3,matrimonio-1000,1862-04-04,1862.0,aucara,julian sanches,jose sanches,leocadia llacsamanta,josefa ramos,tiburcio ramos,rosa quispe,...,rosa,quispe,julian sanches | josefa ramos,julian sanches | josefa,josefa ramos | julian,jose sanches | leocadia llacsamanta,tiburcio ramos | rosa quispe,True,True,True
4,matrimonio-1001,1862-04-04,1862.0,aucara,mariano flores,eucebio flores,maria ynca,maria espinosa,manuel espinosa,hermenejilda luca,...,hermenejilda,luca,mariano flores | maria espinosa,mariano flores | maria,maria espinosa | mariano,eucebio flores | maria ynca,manuel espinosa | hermenejilda luca,True,True,True


MarriageCouples coverage:


has_husband        1.000000
has_wife           0.998836
has_spouse_pair    0.998836
dtype: float64

BURIAL FAMILY

In [13]:
burial_roles = [
    "deceased",
    "father",
    "mother",
    "husband",
    "wife"
]

burial_family = build_event_role_table(
    df=prl,
    source_name="burial",
    allowed_roles=burial_roles,
    output_name="BurialFamily"
)

# Add first and last name parts
for role in burial_roles:
    name_col = f"{role}_name"
    if name_col in burial_family.columns:
        burial_family[f"{role}_first_name"] = burial_family[name_col].apply(first_token)
        burial_family[f"{role}_last_name"] = burial_family[name_col].apply(last_token)

# General spouse column for burial
# If husband_name exists, use it. Otherwise, use wife_name.
burial_family["spouse_name"] = burial_family.apply(
    lambda x: x.get("husband_name") if pd.notna(x.get("husband_name")) 
    else x.get("wife_name") if pd.notna(x.get("wife_name")) 
    else np.nan,
    axis=1
)

burial_family["spouse_persona_id"] = burial_family.apply(
    lambda x: x.get("husband_persona_id") if pd.notna(x.get("husband_persona_id")) 
    else x.get("wife_persona_id") if pd.notna(x.get("wife_persona_id")) 
    else np.nan,
    axis=1
)

# Burial keys
burial_family["burial_parent_pair_key"] = burial_family.apply(
    lambda x: make_key(x.get("father_name"), x.get("mother_name")),
    axis=1
)

burial_family["deceased_parent_key"] = burial_family.apply(
    lambda x: make_key(x.get("deceased_name"), x.get("father_name"), x.get("mother_name")),
    axis=1
)

burial_family["deceased_spouse_key"] = burial_family.apply(
    lambda x: make_key(x.get("deceased_name"), x.get("spouse_name")),
    axis=1
)

burial_family["father_motherfirst_key"] = burial_family.apply(
    lambda x: make_key(x.get("father_name"), x.get("mother_first_name")),
    axis=1
)

burial_family["mother_fatherfirst_key"] = burial_family.apply(
    lambda x: make_key(x.get("mother_name"), x.get("father_first_name")),
    axis=1
)

# Useful flags
burial_family["has_deceased"] = burial_family.get("deceased_name", pd.Series(index=burial_family.index)).notna()
burial_family["has_father"] = burial_family.get("father_name", pd.Series(index=burial_family.index)).notna()
burial_family["has_mother"] = burial_family.get("mother_name", pd.Series(index=burial_family.index)).notna()
burial_family["has_husband"] = burial_family.get("husband_name", pd.Series(index=burial_family.index)).notna()
burial_family["has_wife"] = burial_family.get("wife_name", pd.Series(index=burial_family.index)).notna()
burial_family["has_spouse"] = burial_family["spouse_name"].notna()
burial_family["has_deceased_parent_key"] = burial_family["deceased_parent_key"].notna()
burial_family["has_deceased_spouse_key"] = burial_family["deceased_spouse_key"].notna()

print("BurialFamily preview:")
display(burial_family.head())

print("BurialFamily coverage:")
display(burial_family[[
    "has_deceased",
    "has_father",
    "has_mother",
    "has_husband",
    "has_wife",
    "has_spouse",
    "has_deceased_parent_key",
    "has_deceased_spouse_key"
]].mean().sort_values(ascending=False))


BurialFamily source rows: 5395
BurialFamily unique events: 2121
BurialFamily role counts:


role_standard
deceased    2120
mother      1301
father      1299
wife         343
husband      332
Name: count, dtype: int64

BurialFamily final shape: (2115, 14)
BurialFamily preview:


,event_idno,event_date,event_year,event_place_clean,deceased_name,father_name,husband_name,mother_name,wife_name,deceased_persona_id,...,father_motherfirst_key,mother_fatherfirst_key,has_deceased,has_father,has_mother,has_husband,has_wife,has_spouse,has_deceased_parent_key,has_deceased_spouse_key
0,entierro-1,1846-10-06,1846.0,unknown,julian xavies,NaN,NaN,NaN,mercedes lupa,persona-41678,...,NaN,NaN,True,False,False,False,True,True,False,True
1,entierro-10,1848-11-17,1848.0,unknown,manuela hermosa,juan hermosa,NaN,clemencia manus,NaN,persona-41695,...,juan hermosa | clemencia,clemencia manus | juan,True,True,True,False,False,False,True,False
2,entierro-100,1866-05-04,1866.0,unknown,juan sanchez,juan sanchez,NaN,cipriana quispe,NaN,persona-41909,...,juan sanchez | cipriana,cipriana quispe | juan,True,True,True,False,False,False,True,False
3,entierro-1000,1895-12-30,1895.0,unknown,jacova misaguel unknown,NaN,NaN,NaN,NaN,persona-43648,...,NaN,NaN,True,False,False,False,False,False,False,False
4,entierro-1001,1895-11-27,1895.0,unknown,melchor rojas,NaN,NaN,NaN,NaN,persona-43649,...,NaN,NaN,True,False,False,False,False,False,False,False


BurialFamily coverage:


has_deceased               0.999527
has_mother                 0.614657
has_father                 0.614184
has_deceased_parent_key    0.602837
has_spouse                 0.317258
has_deceased_spouse_key    0.317258
has_wife                   0.161702
has_husband                0.155556
dtype: float64

CHECK DUPLICATE ROLE PROBLEM

In [14]:
def role_multiplicity_report(df):
    """
    Finds cases where the same event has multiple people with the same role.
    Example: one baptism event with two fathers.
    """
    report = (
        df
        .dropna(subset=["event_idno", "role_standard"])
        .groupby(["source_standard", "event_idno", "role_standard"])
        .size()
        .reset_index(name="role_count")
    )
    
    report = report[report["role_count"] > 1].copy()
    return report.sort_values(["source_standard", "event_idno", "role_standard"])


multiplicity = role_multiplicity_report(prl)

print("Events with multiple people in the same role:", multiplicity.shape[0])
display(multiplicity.head(30))

Events with multiple people in the same role: 3221


,source_standard,event_idno,role_standard,role_count
30381,marriage,matrimonio-1,godparent,2
30388,marriage,matrimonio-1,witness,3
30389,marriage,matrimonio-10,godparent,2
30396,marriage,matrimonio-10,witness,3
30397,marriage,matrimonio-100,godparent,2
30403,marriage,matrimonio-100,witness,3
30404,marriage,matrimonio-1000,godparent,2
30411,marriage,matrimonio-1000,witness,2
30412,marriage,matrimonio-1001,godparent,2
30419,marriage,matrimonio-1001,witness,2


source role summary

In [19]:
source_role_summary = (
    prl
    .groupby(["source_standard", "role_standard"])
    .size()
    .reset_index(name="row_count")
    .sort_values(["source_standard", "row_count"], ascending=[True, False])
)

display(source_role_summary)

,source_standard,role_standard,row_count
0,baptism,child,6340
4,baptism,mother,6313
1,baptism,father,6070
3,baptism,godmother,3251
2,baptism,godfather,3012
5,burial,deceased,2120
8,burial,mother,1301
6,burial,father,1299
9,burial,wife,343
7,burial,husband,332


Save Outputs

In [20]:
def coverage_summary(df, table_name):
    """
    Creates a coverage summary for a table.
    Shows how many non-null values each column has.
    """
    rows = []

    for col in df.columns:
        total_count = len(df)
        non_null_count = df[col].notna().sum()
        non_null_pct = non_null_count / total_count if total_count > 0 else 0

        rows.append({
            "table": table_name,
            "column": col,
            "non_null_count": non_null_count,
            "total_count": total_count,
            "non_null_pct": round(non_null_pct, 4)
        })

    return pd.DataFrame(rows)


coverage = pd.concat(
    [
        coverage_summary(baptism_family, "BaptismFamily"),
        coverage_summary(marriage_couples, "MarriageCouples"),
        coverage_summary(burial_family, "BurialFamily")
    ],
    ignore_index=True
)

print("Coverage table created:", coverage.shape)
display(coverage.head())

Coverage table created: (100, 5)


,table,column,non_null_count,total_count,non_null_pct
0,BaptismFamily,event_idno,6338,6338,1.0
1,BaptismFamily,event_date,6338,6338,1.0
2,BaptismFamily,event_year,6338,6338,1.0
3,BaptismFamily,event_place_clean,6338,6338,1.0
4,BaptismFamily,child_name,6338,6338,1.0


In [21]:
baptism_family_path = OUTPUT_DIR / "baptism_family.csv"
marriage_couples_path = OUTPUT_DIR / "marriage_couples.csv"
burial_family_path = OUTPUT_DIR / "burial_family.csv"
coverage_path = OUTPUT_DIR / "family_evidence_coverage.xlsx"

baptism_family.to_csv(baptism_family_path, index=False, encoding="utf-8-sig")
marriage_couples.to_csv(marriage_couples_path, index=False, encoding="utf-8-sig")
burial_family.to_csv(burial_family_path, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(coverage_path, engine="openpyxl") as writer:
    coverage.to_excel(writer, sheet_name="Column_Coverage", index=False)
    source_role_summary.to_excel(writer, sheet_name="Source_Role_Summary", index=False)
    multiplicity.to_excel(writer, sheet_name="Role_Multiplicity_Check", index=False)
    
    baptism_family.head(100).to_excel(writer, sheet_name="BaptismFamily_Sample", index=False)
    marriage_couples.head(100).to_excel(writer, sheet_name="MarriageCouples_Sample", index=False)
    burial_family.head(100).to_excel(writer, sheet_name="BurialFamily_Sample", index=False)

print("Saved files:")
print(baptism_family_path)
print(marriage_couples_path)
print(burial_family_path)
print(coverage_path)

Saved files:
outputs\family_evidence\baptism_family.csv
outputs\family_evidence\marriage_couples.csv
outputs\family_evidence\burial_family.csv
outputs\family_evidence\family_evidence_coverage.xlsx
